# Professional ResNet-50 Training for Mammography

**Production-ready training on 1000 stratified VinDr-Mammo images**

## Architecture:
- ResNet-50 pretrained on ImageNet
- Binary classification (malignant vs benign)
- Proper normalization and preprocessing
- Class-weighted loss for imbalanced data

## Dataset:
- Uses preprocessed PNG images (512×512)
- Stratified sampling: 250 malignant, 750 benign
- Patient-level split to prevent leakage

---

## Step 0: Verify GPU and Environment

In [ ]:
!nvidia-smi

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

import numpy as np
import pandas as pd
import os
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.utils.class_weight import compute_class_weight

print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Training will be VERY slow.")

print("🔍 Checking if images actually exist...\n")

# Prepare metadata
def prepare_metadata(df):
    df = df.copy()
    if 'image_id' not in df.columns:
        if 'png_path' in df.columns:
            df['image_id'] = df['png_path'].apply(lambda x: Path(x).stem)
        elif 'file_path' in df.columns:
            df['image_id'] = df['file_path'].apply(lambda x: Path(x).stem)
    
    if 'patient_id' not in df.columns and 'study_id' in df.columns:
        df['patient_id'] = df['study_id']
    
    if 'breast_id' not in df.columns:
        if 'laterality' in df.columns and 'patient_id' in df.columns:
            df['breast_id'] = df['patient_id'].astype(str) + '_' + df['laterality'].astype(str)
        elif 'patient_id' in df.columns:
            df['breast_id'] = df['patient_id'].astype(str) + '_Unknown'
    
    # Determine image path column and convert .dicom to .png
    if 'png_path' in df.columns:
        df['image_path'] = df['png_path']
    elif 'file_path' in df.columns:
        # Convert .dicom to .png
        df['image_path'] = df['file_path'].str.replace('.dicom', '.png', regex=False)
    
    return df

df = prepare_metadata(df)

# Check sample of images
sample_size = min(50, len(df))
missing = 0
valid = 0

for idx in range(sample_size):
    img_path = os.path.join(PREPROCESSED_DIR, df.iloc[idx]['image_path'])
    if os.path.exists(img_path):
        valid += 1
    else:
        missing += 1
        if missing <= 3:
            print(f"❌ Missing: {img_path}")

print(f"\nChecked {sample_size} images:")
print(f"  Valid: {valid} ({valid/sample_size*100:.1f}%)")
print(f"  Missing: {missing} ({missing/sample_size*100:.1f}%)")

if missing > sample_size * 0.1:
    print("\n❌ ERROR: Too many missing images!")
    print("Please check your preprocessing and file paths.")
    raise FileNotFoundError(f"Missing {missing}/{sample_size} images")

print("\n✅ Images verified")
print(f"✅ Converted .dicom paths to .png paths")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paths
BASE_DIR = '/content/drive/MyDrive/vindr-mammo'
PREPROCESSED_DIR = f'{BASE_DIR}/preprocessed_png_512'
STRATIFIED_CSV = f'{BASE_DIR}/metadata/stratified_selection.csv'
OUTPUT_DIR = '/content/drive/MyDrive/resnet50_training'

# Verify paths exist
print("\n📁 Checking paths...")
print(f"Base directory exists: {os.path.exists(BASE_DIR)}")
print(f"Preprocessed images exist: {os.path.exists(PREPROCESSED_DIR)}")
print(f"Stratified CSV exists: {os.path.exists(STRATIFIED_CSV)}")

if not os.path.exists(STRATIFIED_CSV):
    print("\n❌ ERROR: stratified_selection.csv not found!")
    print(f"Expected at: {STRATIFIED_CSV}")
    print("\nPlease run VinDr_Mammo_Restratify_Downloaded.ipynb first!")
    raise FileNotFoundError("Stratified dataset not found")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"\n✅ All paths verified")

## Step 2: Load Full Stratified Dataset

In [ ]:
print("📊 Loading stratified dataset...\n")
df = pd.read_csv(STRATIFIED_CSV)

print("="*70)
print("DATASET SUMMARY")
print("="*70)
print(f"Total images: {len(df)}")
print(f"\nLabel distribution:")
print(f"  Malignant (1): {(df['label'] == 1).sum()} ({(df['label'] == 1).sum()/len(df)*100:.1f}%)")
print(f"  Benign (0):    {(df['label'] == 0).sum()} ({(df['label'] == 0).sum()/len(df)*100:.1f}%)")

if 'study_id' in df.columns:
    print(f"\nPatients: {df['study_id'].nunique()}")

if 'breast_birads' in df.columns:
    print(f"\nBI-RADS distribution:")
    print(df['breast_birads'].value_counts().sort_index())

print(f"\nColumns: {list(df.columns)}")
print("="*70)

# Verify we have enough data
if len(df) < 100:
    print("\n⚠️  WARNING: Very small dataset! Performance will be limited.")
elif len(df) < 500:
    print("\n⚠️  Small dataset. Will use conservative hyperparameters.")
else:
    print(f"\n✅ Good dataset size ({len(df)} images)")

## Step 3: Verify Images Are Actually Available

In [ ]:
print("🔍 Checking if images actually exist...\n")

# Prepare metadata
def prepare_metadata(df):
    df = df.copy()
    if 'image_id' not in df.columns:
        if 'png_path' in df.columns:
            df['image_id'] = df['png_path'].apply(lambda x: Path(x).stem)
        elif 'file_path' in df.columns:
            df['image_id'] = df['file_path'].apply(lambda x: Path(x).stem)
    
    if 'patient_id' not in df.columns and 'study_id' in df.columns:
        df['patient_id'] = df['study_id']
    
    if 'breast_id' not in df.columns:
        if 'laterality' in df.columns and 'patient_id' in df.columns:
            df['breast_id'] = df['patient_id'].astype(str) + '_' + df['laterality'].astype(str)
        elif 'patient_id' in df.columns:
            df['breast_id'] = df['patient_id'].astype(str) + '_Unknown'
    
    # Determine image path column
    if 'png_path' in df.columns:
        df['image_path'] = df['png_path']
    elif 'file_path' in df.columns:
        df['image_path'] = df['file_path']
    
    return df

df = prepare_metadata(df)

# Check sample of images
sample_size = min(50, len(df))
missing = 0
valid = 0

for idx in range(sample_size):
    img_path = os.path.join(PREPROCESSED_DIR, df.iloc[idx]['image_path'])
    if os.path.exists(img_path):
        valid += 1
    else:
        missing += 1
        if missing <= 3:
            print(f"❌ Missing: {img_path}")

print(f"\nChecked {sample_size} images:")
print(f"  Valid: {valid} ({valid/sample_size*100:.1f}%)")
print(f"  Missing: {missing} ({missing/sample_size*100:.1f}%)")

if missing > sample_size * 0.1:
    print("\n❌ ERROR: Too many missing images!")
    print("Please check your preprocessing and file paths.")
    raise FileNotFoundError(f"Missing {missing}/{sample_size} images")

print("\n✅ Images verified")

## Step 4: Patient-Level Train/Val/Test Split

In [ ]:
print("📊 Creating patient-level train/val/test split...\n")

# Get unique patients
patients = df['patient_id'].unique()
np.random.seed(42)
np.random.shuffle(patients)

# 70/15/15 split
n_train = int(len(patients) * 0.70)
n_val = int(len(patients) * 0.15)

train_patients = set(patients[:n_train])
val_patients = set(patients[n_train:n_train + n_val])
test_patients = set(patients[n_train + n_val:])

# Split dataframe
train_df = df[df['patient_id'].isin(train_patients)].reset_index(drop=True)
val_df = df[df['patient_id'].isin(val_patients)].reset_index(drop=True)
test_df = df[df['patient_id'].isin(test_patients)].reset_index(drop=True)

print("="*70)
print("TRAIN/VAL/TEST SPLIT")
print("="*70)
print(f"\nTrain: {len(train_patients)} patients, {len(train_df)} images")
print(f"  Malignant: {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).sum()/len(train_df)*100:.1f}%)")
print(f"  Benign:    {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).sum()/len(train_df)*100:.1f}%)")

print(f"\nVal: {len(val_patients)} patients, {len(val_df)} images")
print(f"  Malignant: {(val_df['label'] == 1).sum()} ({(val_df['label'] == 1).sum()/len(val_df)*100:.1f}%)")
print(f"  Benign:    {(val_df['label'] == 0).sum()} ({(val_df['label'] == 0).sum()/len(val_df)*100:.1f}%)")

print(f"\nTest: {len(test_patients)} patients, {len(test_df)} images")
print(f"  Malignant: {(test_df['label'] == 1).sum()} ({(test_df['label'] == 1).sum()/len(test_df)*100:.1f}%)")
print(f"  Benign:    {(test_df['label'] == 0).sum()} ({(test_df['label'] == 0).sum()/len(test_df)*100:.1f}%)")
print("="*70)

# Verify no patient overlap
assert len(train_patients & val_patients) == 0, "Patient leakage: train/val overlap!"
assert len(train_patients & test_patients) == 0, "Patient leakage: train/test overlap!"
assert len(val_patients & test_patients) == 0, "Patient leakage: val/test overlap!"
print("\n✅ No patient leakage between splits")

## Step 5: Calculate Class Weights

In [ ]:
# Compute class weights for imbalanced data
labels = train_df['label'].values
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=labels)
pos_weight = class_weights[1] / class_weights[0]

print("⚖️  Class Weights:")
print(f"  Benign (0):    {class_weights[0]:.4f}")
print(f"  Malignant (1): {class_weights[1]:.4f}")
print(f"  Ratio: {pos_weight:.2f}x more weight on malignant samples")
print(f"\nThis addresses {(train_df['label']==0).sum()/(train_df['label']==1).sum():.1f}:1 class imbalance")

## Step 6: Define Dataset with Proper Normalization

In [ ]:
class MammogramDataset(Dataset):
    """Professional dataset with ImageNet normalization."""
    
    def __init__(self, metadata, image_dir, transform=None, augment=False):
        self.metadata = metadata.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.augment = augment
        
        # ImageNet normalization (REQUIRED for pretrained ResNet)
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
        
        # Augmentation for training
        if augment:
            self.aug_transforms = transforms.Compose([
                transforms.RandomAffine(degrees=5, translate=(0.05, 0.05)),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
            ])
    
    def __len__(self):
        return len(self.metadata)
    
    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        
        # Load image
        img_path = os.path.join(self.image_dir, row['image_path'])
        image = Image.open(img_path).convert('L')
        
        # Resize
        if self.transform:
            image = self.transform(image)
        
        # Convert to RGB (repeat grayscale)
        image = image.convert('RGB')
        
        # Convert to tensor
        image = transforms.ToTensor()(image)
        
        # Augmentation (before normalization)
        if self.augment:
            image = self.aug_transforms(image)
        
        # ImageNet normalization
        image = self.normalize(image)
        
        label = int(row['label'])
        image_id = row['image_id']
        
        return image, label, image_id

print("✅ Dataset class defined with ImageNet normalization")

## Step 7: Create DataLoaders

In [ ]:
# Hyperparameters based on dataset size
BATCH_SIZE = 32 if len(train_df) > 500 else 16
NUM_WORKERS = 2

# Transforms
transform = transforms.Resize((224, 224))

# Create datasets
train_dataset = MammogramDataset(
    train_df, PREPROCESSED_DIR, transform=transform, augment=True
)
val_dataset = MammogramDataset(
    val_df, PREPROCESSED_DIR, transform=transform, augment=False
)
test_dataset = MammogramDataset(
    test_df, PREPROCESSED_DIR, transform=transform, augment=False
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

print(f"✅ DataLoaders created:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")
print(f"  Training augmentation: ON")
print(f"  ImageNet normalization: ON")

## Step 8: Define Model

In [ ]:
class ResNet50Binary(nn.Module):
    """ResNet-50 for binary classification."""
    
    def __init__(self, dropout=0.3, freeze_backbone_layers=0.7):
        super().__init__()
        
        # Load pretrained ResNet-50
        resnet = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        
        # Remove final FC layer
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        
        # Freeze early layers
        total_layers = len(list(self.features.parameters()))
        layers_to_freeze = int(total_layers * freeze_backbone_layers)
        
        for i, param in enumerate(self.features.parameters()):
            if i < layers_to_freeze:
                param.requires_grad = False
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2048, 1)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ResNet50Binary(dropout=0.3, freeze_backbone_layers=0.7).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model created:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"  Device: {device}")

## Step 9: Setup Training

In [ ]:
# Loss with class weights
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(device))

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, verbose=True
)

# Metrics
def compute_metrics(preds, labels):
    preds = np.array(preds)
    labels = np.array(labels)
    
    auroc = roc_auc_score(labels, preds)
    pr_auc = average_precision_score(labels, preds)
    brier = brier_score_loss(labels, preds)
    
    return {'auroc': auroc, 'pr_auc': pr_auc, 'brier': brier}

print("✅ Training setup complete:")
print(f"  Loss: BCEWithLogitsLoss (pos_weight={pos_weight:.2f})")
print(f"  Optimizer: AdamW (lr=1e-4, wd=1e-4)")
print(f"  Scheduler: ReduceLROnPlateau")

## Step 10: Training Loop

In [ ]:
MAX_EPOCHS = 100
PATIENCE = 20

history = {
    'train_loss': [],
    'val_auroc': [],
    'val_pr_auc': [],
    'val_brier': []
}

best_val_auroc = 0.0
patience_counter = 0
best_model_path = os.path.join(OUTPUT_DIR, 'best_model.pth')

print("🚀 Starting training...\n")
print("="*70)

for epoch in range(MAX_EPOCHS):
    # ========== TRAINING ==========
    model.train()
    train_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Train]")
    for images, labels, _ in pbar:
        images = images.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)
    
    # ========== VALIDATION ==========
    model.eval()
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS} [Val]  "):
            images = images.to(device)
            logits = model(images).squeeze(1)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            val_preds.extend(probs)
            val_labels.extend(labels.numpy())
    
    # Compute metrics
    val_metrics = compute_metrics(val_preds, val_labels)
    
    history['val_auroc'].append(val_metrics['auroc'])
    history['val_pr_auc'].append(val_metrics['pr_auc'])
    history['val_brier'].append(val_metrics['brier'])
    
    # Print
    print(f"\nEpoch {epoch+1}/{MAX_EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f}")
    print(f"  Val AUROC:  {val_metrics['auroc']:.4f}")
    print(f"  Val PR-AUC: {val_metrics['pr_auc']:.4f}")
    print(f"  Val Brier:  {val_metrics['brier']:.4f}")
    
    # Learning rate scheduling
    scheduler.step(val_metrics['auroc'])
    
    # Save best model
    if val_metrics['auroc'] > best_val_auroc:
        best_val_auroc = val_metrics['auroc']
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'metrics': val_metrics,
        }, best_model_path)
        print(f"  💾 Saved best model (AUROC: {best_val_auroc:.4f})")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f"\n⚠️  Early stopping after {PATIENCE} epochs without improvement")
        break
    
    print("="*70)

print(f"\n✅ Training complete!")
print(f"   Best val AUROC: {best_val_auroc:.4f}")
print(f"   Model saved to: {best_model_path}")

## Step 11: Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load(best_model_path)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("📊 Evaluating on test set...\n")

test_preds = []
test_labels = []

with torch.no_grad():
    for images, labels, _ in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        logits = model(images).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        test_preds.extend(probs)
        test_labels.extend(labels.numpy())

test_metrics = compute_metrics(test_preds, test_labels)

print("="*70)
print("FINAL TEST SET RESULTS")
print("="*70)
print(f"AUROC:  {test_metrics['auroc']:.4f}")
print(f"PR-AUC: {test_metrics['pr_auc']:.4f}")
print(f"Brier:  {test_metrics['brier']:.4f}")
print("="*70)

## Step 12: Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'])
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

axes[1].plot(history['val_auroc'], label='AUROC')
axes[1].plot(history['val_pr_auc'], label='PR-AUC')
axes[1].axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random')
axes[1].set_title('Validation Metrics')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].legend()
axes[1].grid(True)

axes[2].plot(history['val_brier'])
axes[2].set_title('Validation Brier Score')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Brier Score (lower better)')
axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150)
plt.show()

print("✅ Training history saved")

## Summary

✅ **Professional training complete!**

**What was done:**
- Used ALL available stratified images
- Patient-level split (no data leakage)
- ImageNet normalization (CRITICAL)
- Class-weighted loss
- Proper augmentation
- Early stopping and LR scheduling

**Files saved:**
- `{OUTPUT_DIR}/best_model.pth`
- `{OUTPUT_DIR}/training_history.png`

**Next steps:**
- If results still poor, run diagnostics
- Check if you have enough malignant samples
- Consider ensemble or different architecture